In [4]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
months = ['202507', '202508', '202509', '202510', '202511', '202512']
alldata = pd.concat([pd.read_csv(f'CRMLSSold{m}.csv') for m in months], ignore_index=True)

df = alldata[
    (alldata['PropertyType'] == 'Residential')
    & (alldata['PropertySubType'] == 'SingleFamilyResidence')
    & (alldata['ClosePrice'] > 0)
    & (alldata['LivingArea'] > 0)
    & (alldata['BedroomsTotal'] > 0)
    & (alldata['BathroomsTotalInteger'] > 0)
    & (alldata['LotSizeSquareFeet'] > 0)
]
df = df.drop_duplicates(subset='ListingKey')
df = df.dropna(axis=1, how='all')
df.info()

<class 'pandas.DataFrame'>
Index: 65961 entries, 3 to 131918
Data columns (total 70 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   BuyerAgentAOR             65950 non-null  str    
 1   ListAgentAOR              65950 non-null  str    
 2   Flooring                  43132 non-null  str    
 3   ViewYN                    59721 non-null  object 
 4   WaterfrontYN              27 non-null     object 
 5   BasementYN                1581 non-null   object 
 6   PoolPrivateYN             60899 non-null  object 
 7   OriginalListPrice         65830 non-null  float64
 8   ListingKey                65961 non-null  int64  
 9   ListAgentEmail            65728 non-null  str    
 10  CloseDate                 65961 non-null  str    
 11  ClosePrice                65961 non-null  float64
 12  ListAgentFirstName        65490 non-null  str    
 13  ListAgentLastName         65954 non-null  str    
 14  Latitude             

Dropping Columns:
- Dropped Listing Key, Names, emails, MLS, UnparsedAddress, StreetNumberNumeric: not attribute predictors
- Dropped PropertyType, PropertySubType, StateOrProvince, MlsStatus - Only showed one value, not helpful
- Dropped LostSizeAcres, LotSizeArea - used SizeSquareFootInstead

In [14]:
USE_LIST_PRICE = False

df = df.copy()
id_admin = [
    'ListingKey', 'ListingKeyNumeric', 'ListingId', 'UnparsedAddress', 'StreetNumberNumeric',
    'BuyerAgentAOR', 'ListAgentAOR', 'BuyerOfficeAOR', 'ListAgentEmail',
    'ListAgentFirstName', 'ListAgentLastName', 'ListAgentFullName',
    'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentMlsId',
    'CoListAgentFirstName', 'CoListAgentLastName', 'CoBuyerAgentFirstName',
    'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
]
constant = ['PropertyType', 'PropertySubType', 'StateOrProvince', 'MlsStatus']
leakage = ['DaysOnMarket', 'ContractStatusChangeDate', 'PurchaseContractDate']
redundant = ['LotSizeAcres', 'LotSizeArea']

for c in constant:
    if c in df:
        print(f'{c}: {df[c].nunique()} unique value(s)')

df = df.drop(columns=id_admin + constant + leakage + redundant, errors='ignore')
print(df.shape)

PropertyType: 1 unique value(s)
PropertySubType: 1 unique value(s)
StateOrProvince: 3 unique value(s)
MlsStatus: 1 unique value(s)
(65961, 40)


Removed Missing Values

In [15]:
df = df.dropna(subset=['Latitude', 'Longitude'])

def yn_to_int(s):
    return (s.astype(str).str.strip().str.lower()
             .map({'true': 1, 'false': 0, 'yes': 1, 'no': 0, 'y': 1, 'n': 0, '1': 1, '0': 0}))

for c in ['WaterfrontYN', 'BasementYN']:
    df[c] = yn_to_int(df[c]).fillna(0).astype(int)

for c in ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']:
    v = yn_to_int(df[c])
    df[c + '_missing'] = v.isna().astype(int)
    df[c] = v.fillna(0).astype(int)

print(df['AssociationFeeFrequency'].value_counts(dropna=False))

AssociationFeeFrequency
NaN             49676
Monthly         14296
Annually         1268
Quarterly         609
SemiAnnually      107
Name: count, dtype: int64


Feature Engineering:
- CloseMonth - got from CloseDate

In [16]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df['CloseMonth'] = df['CloseDate'].dt.to_period('M').astype(str)

print(df['CloseMonth'].value_counts().sort_index())
print(df.shape)

CloseMonth
2025-07    11897
2025-08    11220
2025-09    11218
2025-10    11782
2025-11     9576
2025-12    10263
Name: count, dtype: int64
(65956, 46)


Saved CSV

In [17]:
df.to_csv('cleaned_sfr_2025H2.csv', index=False)